# Exoplanet Detection using a Neural Network

This notebook combines the complete workflow:
1. Load data
2. Preprocess
3. Build model
4. Train
5. Evaluate

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import RandomOverSampler


## Load Dataset

In [ ]:
dataset_path = "../data"

train = pd.read_csv(f"{dataset_path}/exoTrain.csv")
test = pd.read_csv(f"{dataset_path}/exoTest.csv")

X_train = train.iloc[:,1:]
y_train = train.iloc[:,0] - 1

X_test = test.iloc[:,1:]
y_test = test.iloc[:,0] - 1

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

ros = RandomOverSampler(random_state=42)
X_train, y_train = ros.fit_resample(X_train, y_train)

print(X_train.shape)
print(X_test.shape)


## Build Neural Network

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()


## Train Model

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=1
)

model.save("../models/exoplanet_detector.keras")


## Evaluate on exoTest.csv

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
